Subject: ST 554 - Final Project

Name: Franklin Zhou

Date: 4/19/2026

# Fitting Your Model (50 pts)

**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**

- The file `power_ml_data.csv` is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the `pd.read_csv()` function.
- Convert this to a spark data frame
- We are going to treat the `Power_Zone_3` variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the `Power_Zone_3` reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [2]:
# Load packages and initiate spark session
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [4]:
# Read data
ml_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df = spark.createDataFrame(ml_data) # convert to spark sql data frame
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read in) with the steps below. 

The transformations below should each use an `MLlib` function that can be put into a pipeline

- The Hour column is likely not stored as a `DoubleType`. If it is not, use an SQL transformer to cast the variable as a `DoubleType`


In [5]:
# Load packages
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, OneHotEncoder, StringIndexer, PCA
from pyspark.ml import Pipeline


In [6]:
# Check the schema
df.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [7]:
# Cast Hour to Double Type 
cast_sql = SQLTransformer(
    statement = """
        SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double
        FROM __THIS__
    """
)

- Binarize the `Hour` column based on the column being less than 6.5 or not (night vs day essentially)

In [8]:
# Binarize Hour_Double: 1 if Hour < 6.5 (night), 0 otherwise (day).
binarizer = Binarizer(
    inputCol = "Hour_Double",
    outputCol = "Hour_Bin",
    threshold = 6.5
)

- One-hot encode the Month column

In [9]:
# Cast Month to string first via a SQLTransformer, then index and encode.
cast_month_sql = SQLTransformer(
    statement = "SELECT *, CAST(Month AS STRING) AS Month_str FROM __THIS__"
)

# StringIndexer maps each month a numeric index
month_indexer = StringIndexer(
    inputCol = "Month_str",
    outputCol = "Month_idx"
)

# OneHotEncoder converts numeric index to binary vector
month_encoder = OneHotEncoder(
    inputCol = "Month_idx",
    outputCol = "Month_vector"
)

- Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns.

     - To do this, I first used a `VectorAssembler()` call to place these variables in a column together for use with the `PCA()` estimator.
    
     - Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.
    
    - We’ll use two PCs in our transformation.

In [10]:
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
    outputCol = "PCA_input"
)

pca = PCA(
    k = 2, 
    inputCol = "PCA_input", 
    outputCol = "PCA_features"
)

- Rename your response variable as `label`

In [11]:
# Rename Power_Zone_3 to label
label_sql = SQLTransformer(
    statement = "SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

- Use VectorAssembler() to put your predictors into a features. Use the
     - two fitted PCA features
     - binary `Hour` variable
     - `Power_Zone_1`
     - `Power_Zone_2`
     - `Month` indicator variables


In [12]:
# Combine all predictors into the 'features' vector column.
features_assembler = VectorAssembler(
    inputCols = ["PCA_features", "Hour_Bin", "Power_Zone_1", "Power_Zone_2", "Month_vector"],
    outputCol = "features"
)

- This ends the pipeline of transformations!

- Now you’ll then use the `CrossValidator()` function and the LinearRegression() function to fit an elastic net model.

    - You should do the following grid for the regParam and elasticNetParam: All combinations of
    
        - regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
        - elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
- Now fit the model using 5-fold CV with `rmse` as your criterion!
        

In [13]:
# Load packages
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [14]:
# Setup LinearRegression instance
lr = LinearRegression()

# Setup parameters grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Setup pipeline
transformation_pipeline = Pipeline(stages=[cast_sql, binarizer, cast_month_sql, month_indexer, month_encoder, pca_assembler, pca, label_sql, features_assembler, lr])

# Create cross validation instance
crossval_lr = CrossValidator(estimator = transformation_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5)

In [15]:
# Fit the cv model
cv_model = crossval_lr.fit(df)

26/04/20 20:42:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/20 20:42:15 WARN Instrumentation: [0133ad73] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:18 WARN Instrumentation: [d4518763] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:20 WARN Instrumentation: [ec0de45a] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:21 WARN Instrumentation: [4c72db74] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:23 WARN Instrumentation: [9b098424] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:24 WARN Instrumentation: [f0e869d8] regParam is zero, which might cause numerical instability and overfitting.
26/04/20 20:42:25 WARN Instrumentation: [806f726c] regP

- Report the optimal values chosen for the tuning parameters

- Report the CV error

In [17]:
# Create a list contains RMSE value associate with parameters value
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cv_model.avgMetrics[i], paramGrid[i].values()])

import numpy as np

# Convert to numpy array 
arrange = np.array(my_list)

# Sort by RMSE value
my_list_sorted = arrange[arrange[:, 0].argsort()]

# Print top 5 rows
print(my_list_sorted[:5])

[[2147.7172958526485 dict_values([0.75, 0.05])]
 [2147.7173788117725 dict_values([0.99, 0.05])]
 [2147.717409723099 dict_values([0.98, 0.05])]
 [2147.717410953494 dict_values([1.0, 0.05])]
 [2147.717433788893 dict_values([0.9, 0.1])]]


From the output we find that the minumum RMSE is 2147.7172958526485 while the best `regParam` value is 0.75 and the best `elasticNetParam` value is 0.05.

In [19]:
# Another way to extract the value
best_lr = cv_model.bestModel.stages[-1]

# Retrieve optimal tuning parameters
print("Best regParam value is:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam value is:", best_lr._java_obj.getElasticNetParam())

# CV RMSE 
print("CV RMSE:", min(cv_model.avgMetrics))

Best regParam value is: 0.75
Best elasticNetParam value is: 0.05
CV RMSE: 2147.7172958526485


- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and evaluating on the entire training set

In [20]:
# Now cv_model is a transformer with "best model" as default.
train_predictions = cv_model.transform(df)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(train_predictions)
print("The training RMSE value is:", training_rmse)

The training RMSE value is: 2147.09756941097


- Take the outputted transformations from the model (the predictions) and create a `residual` column (`label` - `prediction`). The `.withColumn()` method is handy here. Print out a data frame with these `residual`s, the `label` column, and the `prediction`s

In [21]:
from pyspark.sql.functions import col
# create column residual = label - prediction
res_df = train_predictions.withColumn("residual", col("label") - col("prediction")).select("label","prediction","residual")

res_df.show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20876.056487881455| -635.092627881455|
|20131.08434|18659.780441777348|1471.3038982226535|
|19668.43373|18204.414617418803| 1464.019112581198|
|18899.27711|17590.404717709935|1308.8723922900645|
|18442.40964|  16997.1975258365|1445.2121141635034|
|18130.12048|16517.672857316004| 1612.447622683998|
|17945.06024|16093.305472662243|1851.7547673377558|
|17459.27711|15722.832038428522|1736.4450715714775|
|17025.54217|15271.264023800832| 1754.278146199169|
|16794.21687|14938.641983195212|1855.5748868047885|
+-----------+------------------+------------------+
only showing top 10 rows


## Streaming Part (40 pts)

There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv

Download this file and store it where your `.py` file you’ll create can find it. We’ll be randomly sampling rows from this to output to `.csv` files that you’ll be reading in.

### Reading a Stream

- We’re going to read in a stream in the form of `.csv` files. Create a folder where you will be sending your `.csv` files.
- Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)
- Set up the `readStream`. Be sure to add `header = True` as you’ll likely be outputting files with a header and we don’t need to read that in.

In [22]:
data_schema = df.schema

In [23]:
# read stream data from Final_Project/stream_folder folder
stream_df = spark.readStream.option("header", True).schema(data_schema).csv("stream_folder")

### Transform/Aggregation Step

- Now, we’ll do two separate things on the stream and join them together:
    - With your stream, use your model transformer to obtain predictions from the incoming data. On the resulting predictions also create a `residual` column as noted in the previous section (return only the `label`, `prediction` and `residual` columns from this part)
    - We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called `label`.
    - Now join your above transform with this stream based on the `label` variable which should be common to both!
    - Note 1: This is a little silly, but I want you to join two transformations of the stream and I don’t want things to get too crazy
    - Note 2: Each data frame is created from the same stream of data! You don’t need two streams, you can use the same stream and just do two separate transformations on it, combining it with a `.join()` method from one of the SQL style data frames you are dealing with (as we discussed in the notes)

In [24]:
# use model transformer to obtain predictions from the incoming data
stream_predictions = cv_model.transform(stream_df)

# Add residual column and keep only the three required columns
stream_with_residuals = stream_predictions.withColumn("residual", col("label") - col("prediction")).select("label", "prediction", "residual")

In [25]:
# Transformation 2: Rename the response variable
stream_label = stream_df.withColumnRenamed("Power_Zone_3", "label")

# Inner join the two stream transformations by label
joined_stream = stream_with_residuals.join(stream_label, on = "label", how = "inner")

### Writing Step

- Now write your stream to the console using the append output mode.

- Start the query!

In [40]:
# Start the query
stream_query = joined_stream \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/20 21:08:41 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-25fdfc7a-6841-4905-a05c-f055f86abc7b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 21:08:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|    15744.0|15624.971701038601| 119.02829896139883|      11.69|    83.3|     0.081|                0.037|        0.111| 23957.63186| 13644.80652|    4|   5|
|15437.21489|17231.311177083793|-1794.0962870837939|      14.64|    78.8|     0.086|                0.048|        0.145| 38211.40684| 31161.70604|   12|  19|
|22262.76151| 25592.23440485901| -3329.472894859009|      23.89|    86.3|     4.915|                0.073|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20746.83386|21127.883037136256|-381.04917713625764|      24.21|    92.9|     4.927|                0.121|        0.115| 28416.42619| 18813.51637|    8|   4|
| 20776.9279|21305.283581480897|  -528.355681480898|      27.02|   53.92|     4.902|                0.113|        0.089| 28333.31853| 20942.34424|    8|   4|
|11426.72065|12747.535180335308|-1320.8145303353085|      18.56|    89.0|     4.918|                0.048|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11450.32258|11873.324498467067|-423.0019184670673|      11.31|    84.9|     0.091|                21.94|        19.23| 23321.87234| 15010.97561|    3|   7|
|    20198.4| 21019.96915583353|-821.5691558335275|      28.38|   28.82|     4.921|                678.9|        40.98| 37179.33775| 22928.48233|    6|  16|
|14385.29111|15074.091596274997|-688.8004862749967|      23.17|   69.06|     4.929|                471.3|        38.32

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15863.22581| 17560.78136634841|-1697.5555563484086|      16.63|   57.33|     0.083|                309.5|        285.8| 33934.97872| 21095.12195|    3|  15|
|21025.47692|21642.329830969007| -616.8529109690062|      30.24|   26.15|     4.915|                495.9|        148.7| 38043.97351|  23003.3264|    6|  17|
|9564.984802| 8559.350202614354| 1005.6345993856467|      22.49|   58.69|     0.084|                 0.94|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19288.76382| 16578.16440238409|  2710.599417615911|      13.75|    70.4|     0.074|                 86.1|         85.8| 30172.88136| 20275.98784|    2|  12|
|24156.55385|24767.740577729797|  -611.186727729797|      26.07|   37.29|     4.922|                224.3|        239.2| 42049.27152|  26397.5052|    6|  19|
|8485.349544| 7358.739321972575| 1126.6102220274252|      12.38|    87.4|     4.911|                3.068|       

In [ ]:
# Stop the query
stream_query.stop()

# Reference

https://share.google/aimode/nADjDIlL2cBxKdAkP

https://share.google/aimode/e89ejefNn6fCYnS3D

https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html

https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html